# Student Dropout and Academic Success Prediction
**Aim:** Explore student outcomes and build a multiclass logistic regression model for Dropout, Enrolled, and Graduate.

**Dataset:** Upload the UCI student dataset file. It must contain a `Target` column. This notebook produces results only after the dataset is uploaded and the cells are run.

**Responsible use:** Predictions should support further assessment and voluntary support, not be the sole basis for decisions about students.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
print("Libraries imported.")

## 1. Load dataset
In Google Colab, upload the CSV/Excel file when prompted. In local Jupyter, enter its path.

In [ ]:
def load_data():
    try:
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        filename = next(iter(uploaded))
    except ImportError:
        filename = input("Enter CSV/XLSX filename or path: ").strip()
    if filename.lower().endswith(".csv"):
        return pd.read_csv(filename, sep=None, engine="python")
    if filename.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(filename)
    raise ValueError("Please upload a CSV or Excel file.")

df = load_data()
df.columns = [str(c).strip() for c in df.columns]
print("Shape:", df.shape)
print(df.columns.tolist())
display(df.head())

## 2. Inspect data and target

In [ ]:
print("Missing values:")
display(df.isna().sum().sort_values(ascending=False).head(15).to_frame("missing"))
print("Duplicate rows:", int(df.duplicated().sum()))
if "Target" not in df.columns:
    raise ValueError("Column 'Target' not found. Please upload the UCI student outcome dataset.")
df = df.drop_duplicates().dropna(subset=["Target"]).copy()
display(df["Target"].value_counts().to_frame("number_of_students"))
df["Target"].value_counts().plot(kind="bar", title="Distribution of Student Outcomes")
plt.xlabel("Outcome"); plt.ylabel("Number of students"); plt.xticks(rotation=0); plt.tight_layout(); plt.show()

## 3. Explore numeric variables

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
display(df[numeric_cols].describe().T)
for col in ["Age at enrollment", "Admission grade"]:
    if col in df.columns:
        plt.figure(figsize=(7,4))
        plt.hist(df[col].dropna(), bins=20, edgecolor="black")
        plt.title(f"{col} distribution"); plt.xlabel(col); plt.ylabel("Count")
        plt.tight_layout(); plt.show()
if "Admission grade" in df.columns:
    df.boxplot(column="Admission grade", by="Target", figsize=(8,4))
    plt.title("Admission Grades by Outcome"); plt.suptitle("")
    plt.xlabel("Outcome"); plt.ylabel("Admission grade"); plt.tight_layout(); plt.show()

## 4. Optional hypothesis test: first-semester grades

In [ ]:
grade_col = "Curricular units 1st sem (grade)"
if grade_col in df.columns:
    from scipy.stats import f_oneway
    groups = [g[grade_col].dropna().values for _, g in df.groupby("Target") if g[grade_col].dropna().shape[0] > 1]
    if len(groups) >= 2:
        stat, p = f_oneway(*groups)
        print("ANOVA F-statistic:", round(float(stat), 4))
        print("p-value:", p)
        print("At alpha=0.05:", "at least one group mean differs" if p < 0.05 else "insufficient evidence of a mean difference")
    else:
        print("Not enough groups for ANOVA.")
else:
    print(f"Skipped: '{grade_col}' not present.")

## 5. Train a multiclass logistic regression model
This baseline excludes columns whose names indicate second-semester data. Confirm which features are available at the prediction stage before using the model.

In [ ]:
X = df.drop(columns=["Target"]).copy()
y = df["Target"].astype(str)
X = X.dropna(axis=1, how="all")
drop_cols = [c for c in X.columns if "2nd sem" in c.lower() or "second semester" in c.lower()]
X = X.drop(columns=drop_cols, errors="ignore")
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols, errors="ignore")
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in X.columns if c not in num_cols]
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))])
prep = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])
stratify_y = y if y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=stratify_y)
model = Pipeline([("preprocess", prep), ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Model trained.")

## 6. Evaluate model

In [ ]:
print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print(classification_report(y_test, pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, pred, xticks_rotation=45)
plt.title("Confusion Matrix"); plt.tight_layout(); plt.show()

## 7. Interpretation and limitations
- Review class-wise precision, recall, F1-score and the confusion matrix; accuracy alone can hide poor performance for smaller classes.
- Statistical association does not prove causation.
- Check fairness, privacy, data quality, and performance on later student cohorts.
- Use predictions as support indicators only, with human review.
- Include actual values from the executed notebook in the final report; do not invent results.